# 1.3) Data Structures and Control Flow

This subchapter assembles the everyday machinery of python: the four built-in containers (list, tuple, set, dict), the statements that drive decisions and repetition, and the functions and error handling that keep code organised and safe. We carry one example throughout — a small catalogue of weather stations, each with an elevation and a few daily mean temperatures — and close with a generated-code bug that is silent until the function is called a second time.

:::{admonition} Learning objectives
:class: tip
- Choose between list, tuple, set, and dict by how the data will be accessed.
- Read and update nested structures, and access dict values safely with .get.
- Drive logic with if/elif/else and repetition with for/while, using break and continue.
- Pair and index sequences with zip and enumerate, and transform them with comprehensions.
- Write pure, type-hinted functions with default and keyword arguments.
- Validate physical preconditions with raise, and handle expected failures with try/except.
:::

## Lists, tuples, and sets

A list is an ordered, mutable sequence — the default container for a series of measurements. A tuple is an immutable, usually fixed-size record, good for things like a coordinate pair. A set holds unique items with fast membership tests but no order.

In [1]:
# a list: ordered, mutable sequence of daily mean temperatures (°C)
readings_celsius = [-2.3, -1.1, 0.4, 1.2, -0.8]
readings_celsius.append(-3.1)          # lists can grow
print(readings_celsius[0], readings_celsius[-1], len(readings_celsius))
print(readings_celsius[1:3])           # a slice: items at index 1 and 2

# a tuple: immutable record, here a (lat, lon) coordinate in degrees
station_coords = (46.5475, 7.9853)
lat, lon = station_coords              # unpacking into two names
print(lat, lon)

# a set: unordered, unique; membership tests are fast
flags = {"ok", "ok", "suspect", "ok"}
print(sorted(flags), "suspect" in flags)   # duplicates collapse; sorted() for stable display

-2.3 -3.1 6
[-1.1, 0.4]
46.5475 7.9853
['ok', 'suspect'] True


## Dictionaries and nested structures

A dict maps keys to values — ideal for lookup by name or id. Real datasets are usually *nested*: here a dict of station codes, each mapping to a dict of attributes, one of which is itself a list of readings.

In [2]:
stations = {
    "JFJ": {"name": "Jungfraujoch", "elevation_m": 3571,
            "readings_celsius": [-2.3, -1.1, 0.4, 1.2, -0.8]},
    "BAS": {"name": "Basel-Binningen", "elevation_m": 316,
            "readings_celsius": [6.1, 7.4, 5.9, 8.2, 6.8]},
    "LUG": {"name": "Lugano", "elevation_m": 273,
            "readings_celsius": [9.4, 10.2, 8.8, 11.1, 9.9]},
}

# keyed lookup, and safe lookup with a default
print(stations["JFJ"]["name"], stations["JFJ"]["elevation_m"])
print(stations.get("ZRH", {}).get("name", "unknown"))   # 'ZRH' is absent

# .items() yields (key, value) pairs for iteration
for code, info in stations.items():
    print(code, info["name"], len(info["readings_celsius"]), "readings")

Jungfraujoch 3571
unknown
JFJ Jungfraujoch 5 readings
BAS Basel-Binningen 5 readings
LUG Lugano 5 readings


:::{admonition} Computational-thinking fundamental: match the structure to the access pattern
:class: important
The right container is the one whose strengths fit how you will use the data. Use a list for an ordered sequence you iterate or index, a dict for keyed lookup by name or id, a set for membership tests and uniqueness, and a tuple for a small fixed record. Choosing well makes the rest of the code shorter and faster; choosing badly forces awkward workarounds later.
:::

## Decisions: if / elif / else

Conditionals select which block runs. The branches are tried in order, and the first true one wins.

In [3]:
temp_celsius = 6.1
if temp_celsius < 0.0:
    category = "freezing"
elif temp_celsius < 10.0:
    category = "cold"
elif temp_celsius < 20.0:
    category = "mild"
else:
    category = "warm"
print(category)

cold


## Repetition: for, while, break, continue, enumerate, zip

A `for` loop iterates a sequence; a `while` loop runs until a condition fails. `break` exits a loop early, `continue` skips to the next iteration. `enumerate` pairs each item with its index, and `zip` walks two sequences in lockstep.

In [4]:
readings_celsius = [6.1, 7.4, 5.9, 8.2, 6.8]
dates = ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]

# enumerate gives index + value; zip pairs two sequences
for i, (date, temp) in enumerate(zip(dates, readings_celsius)):
    print(i, date, temp)

# while with break: stop at the first day above 8 °C
day = 0
while day < len(readings_celsius):
    if readings_celsius[day] > 8.0:
        print("first warm day:", dates[day])
        break
    day += 1

# continue: skip missing values represented as None
mixed = [6.1, None, 5.9, None, 6.8]
total, n = 0.0, 0
for value in mixed:
    if value is None:
        continue
    total += value
    n += 1
print("mean of present values:", round(total / n, 2))

0 2024-01-01 6.1
1 2024-01-02 7.4
2 2024-01-03 5.9
3 2024-01-04 8.2
4 2024-01-05 6.8
first warm day: 2024-01-04
mean of present values: 6.27


## Comprehensions

A comprehension builds a list or dict in one expression, optionally with a filter. It is the idiomatic replacement for a short loop that only transforms or selects.

In [5]:
readings_celsius = [6.1, 7.4, 5.9, 8.2, 6.8]

# list comprehension: convert each value to kelvin
readings_kelvin = [t + 273.15 for t in readings_celsius]
print([round(t, 2) for t in readings_kelvin])   # round only for clean display

# list comprehension with a filter: keep days above the mean
mean_celsius = sum(readings_celsius) / len(readings_celsius)
warm = [t for t in readings_celsius if t > mean_celsius]
print(round(mean_celsius, 2), warm)

# dict comprehension: station code -> mean temperature
station_means = {
    code: sum(info["readings_celsius"]) / len(info["readings_celsius"])
    for code, info in stations.items()
}
print({code: round(m, 2) for code, m in station_means.items()})

[279.25, 280.55, 279.05, 281.35, 279.95]
6.88 [7.4, 8.2]
{'JFJ': -0.52, 'BAS': 6.88, 'LUG': 9.88}


:::{admonition} Quick exercise: warmest reading per station
:class: note
Using the `stations` dict, build a dict comprehension mapping each station code to its warmest reading (°C).
:::

:::{admonition} Solution
:class: note dropdown
```python
warmest = {code: max(info["readings_celsius"]) for code, info in stations.items()}
print(warmest)
```
:::

:::{admonition} Quick exercise: daily differences with zip
:class: note
The JFJ and BAS readings cover the same five days. Use `zip` to build a list of daily differences, BAS minus JFJ (°C), rounded to two decimals.
:::

:::{admonition} Solution
:class: note dropdown
```python
jfj = stations["JFJ"]["readings_celsius"]
bas = stations["BAS"]["readings_celsius"]
diffs = [round(b - j, 2) for j, b in zip(jfj, bas)]
print(diffs)
```
:::

## Functions: pure, type-hinted, with defaults

A function names a reusable block. A *pure* function depends only on its arguments and has no side effects, which makes it easy to test and reason about. Type hints document the intended types; default arguments supply sensible fallbacks that callers can override by keyword.

In [6]:
def mean(values: list[float]) -> float:
    # pure: output depends only on the input
    return sum(values) / len(values)

def classify_temperature(temp_celsius: float, mild_max: float = 20.0) -> str:
    # mild_max has a default; callers may override it by keyword
    if temp_celsius < 0.0:
        return "freezing"
    elif temp_celsius < 10.0:
        return "cold"
    elif temp_celsius < mild_max:
        return "mild"
    else:
        return "warm"

print(round(mean([6.1, 7.4, 5.9, 8.2, 6.8]), 2))
print(classify_temperature(15.0))                 # default mild_max = 20 -> mild
print(classify_temperature(15.0, mild_max=12.0))  # override -> warm

6.88
mild
warm


## Error handling: raise and try / except

Use `raise` to reject input that violates a physical or logical precondition, and `try/except` to handle an expected failure without crashing the whole program.

In [7]:
def to_kelvin(temp_celsius: float) -> float:
    if temp_celsius < -273.15:
        raise ValueError(f"temperature {temp_celsius} °C is below absolute zero")
    return temp_celsius + 273.15

for value in [20.0, -300.0]:
    try:
        print(round(to_kelvin(value), 2))
    except ValueError as err:
        print("skipped:", err)

293.15
skipped: temperature -300.0 °C is below absolute zero


## When generated code lies: the shared default list

ai assistants reproduce common idioms — including common footguns. The function below collects readings above a threshold into a list. It works perfectly the first time it is called, then quietly returns wrong results.

In [8]:
def collect_warm_days(readings_celsius, threshold=0.0, out=[]):
    # append days warmer than the threshold (as an assistant returned it)
    for r in readings_celsius:
        if r > threshold:
            out.append(r)
    return out

print("JFJ warm days:", collect_warm_days([-2.3, -1.1, 0.4, 1.2, -0.8]))
print("BAS warm days:", collect_warm_days([6.1, 7.4, 5.9, 8.2, 6.8]))

JFJ warm days: [0.4, 1.2]
BAS warm days: [0.4, 1.2, 6.1, 7.4, 5.9, 8.2, 6.8]


:::{admonition} Diagnosis: a mutable default argument
:class: warning
The second call returns BAS's warm days *plus* JFJ's. A default argument is evaluated once, when the function is defined, so `out=[]` creates a single list that is shared by every call that does not pass its own. Each call appends to the same object. The fix is to default to `None` and build a fresh list inside the function.
:::

In [9]:
def collect_warm_days(readings_celsius, threshold=0.0, out=None):
    if out is None:
        out = []                       # a fresh list on every plain call
    for r in readings_celsius:
        if r > threshold:
            out.append(r)
    return out

print("JFJ warm days:", collect_warm_days([-2.3, -1.1, 0.4, 1.2, -0.8]))
print("BAS warm days:", collect_warm_days([6.1, 7.4, 5.9, 8.2, 6.8]))

JFJ warm days: [0.4, 1.2]
BAS warm days: [6.1, 7.4, 5.9, 8.2, 6.8]


:::{admonition} Going deeper: generators and lazy evaluation
:class: seealso dropdown
A generator produces values on demand instead of materialising the whole list — useful for large or streamed data.

```python
# generator expression: parentheses instead of brackets; nothing computed yet
kelvin = (t + 273.15 for t in readings_celsius)
print(next(kelvin))            # first value, produced on demand

# generator function with yield
def running_total(values):
    total = 0.0
    for v in values:
        total += v
        yield total            # emit the partial sum each step
```

Generators are single-pass and lazy; wrap with `list(...)` only when you truly need every value at once.
:::

:::{admonition} Going deeper: match / case
:class: seealso dropdown
structural pattern matching (python 3.10+) reads cleanly when branching on the value or shape of data.

```python
def describe(flag):
    match flag:
        case "ok":
            return "use as is"
        case "suspect" | "missing":
            return "needs review"
        case _:
            return "unknown flag"
```

The final `case _` is the catch-all, like `else`.
:::

:::{admonition} Going deeper: lambda and scope (LEGB)
:class: seealso dropdown
A `lambda` is a small anonymous function, handy as a sort key:

```python
# station codes sorted by elevation, highest first
by_elevation = sorted(stations, key=lambda code: stations[code]["elevation_m"], reverse=True)
```

names resolve by the LEGB rule — Local, then Enclosing, then Global, then Built-in. A name assigned inside a function is local unless declared `global` or `nonlocal`, which is one more reason pure functions that avoid touching globals are easier to reason about.
:::

:::{admonition} Takeaways
:class: danger
- Lists are ordered and mutable, tuples are immutable records, sets hold unique items, dicts map keys to values; pick by access pattern.
- `.get(key, default)` reads a dict safely; `.items()` iterates key–value pairs.
- `enumerate` gives index with value, `zip` pairs sequences, and comprehensions filter and transform in one expression.
- Pure, type-hinted functions with explicit arguments are easier to test and reuse.
- Never use a mutable default argument (`out=[]`): it is created once and shared across calls. Default to `None` and build a fresh object inside.
- `raise` on a violated precondition; handle expected failures with `try/except`.
:::

## Resources

- [Automate the Boring Stuff with Python](https://automatetheboringstuff.com/) — free online; the chapters on flow control, functions, lists, and dictionaries map directly onto this subchapter.
- [Software Carpentry — Programming with Python](https://swcarpentry.github.io/python-novice-inflammation/) — a research-oriented introduction to lists, loops, conditionals, and functions.